### Import Package

In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from tensorflow.keras.callbacks import ModelCheckpoint
from transformers import AutoTokenizer
from transformers import create_optimizer
from transformers import TFAutoModelForSequenceClassification
from transformers import DataCollatorWithPadding

os.environ["TOKENIZERS_PARALLELISM"] = "false" # Stop Warnings

2022-05-31 23:03:59.945113: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-05-31 23:03:59.945132: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "
/home/feng/.local/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read and Split Dataset

In [2]:
training_file = "dataset/train.csv"
test_file = "dataset/test.csv"
output_dir1 = './model2_outputs'
output_dir2 = './model3_outputs'
batch_size = 16

In [3]:
df = load_dataset('csv', data_files = [training_file])
df = df['train'].train_test_split(test_size = 0.1)
df['valid'] = df['test']
df['test'] = load_dataset('csv', data_files = [test_file])['train']

Using custom data configuration default-dacd6034bd90bc38
Reusing dataset csv (/home/feng/.cache/huggingface/datasets/csv/default-dacd6034bd90bc38/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519)
100%|██████████| 1/1 [00:00<00:00, 1135.44it/s]
Using custom data configuration default-60f927a822212e41
Reusing dataset csv (/home/feng/.cache/huggingface/datasets/csv/default-60f927a822212e41/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519)
100%|██████████| 1/1 [00:00<00:00, 255.16it/s]


In [4]:
pd.DataFrame(df['train'])

,id,keyword,location,text,target
0,6594,inundated,Bristol,Hi @FionaGilbert_ sorry for the delay. Slightl...,0
1,1649,bombing,"Overland Park, KS",Japan Marks 70th Anniversary of Hiroshima Atom...,1
2,6627,inundated,Pontefract UK,@LEDofficial1 As you can imagine we're inundat...,1
3,2909,danger,None,The sign-up is open for the FALLING FOR DANGER...,1
4,8278,rioting,None,@aelinrhee a group of mascara smeared girls ri...,1
...,...,...,...,...,...
6846,4610,emergency%20services,"British Columbia, Canada",#veterans VET Act would ensure every military ...,0
6847,5203,fatalities,None,No UK train accident fatalities for 8th year r...,1
6848,10585,wounded,None,Gunmen kill four in El Salvador bus attack: Su...,1
6849,10660,wounds,United States,Having your wounds kissed by Someone who doesn...,0


In [5]:
pd.DataFrame(df['valid'])

,id,keyword,location,text,target
0,9510,terrorist,Loading...,Anyone missing their license plate? Two stolen...,1
1,6530,injuries,None,Diego Costa needs to stop getting injuries urg,0
2,7936,rainstorm,Thailand Malaysia Indonesia,Nigeria: Rainstorm Destroys 600 Houses in Yobe...,1
3,4170,drown,None,@CortneyMo_ put this in Detroit niggas gone be...,0
4,1041,bleeding,None,Apparently if you're bleeding people look at y...,0
...,...,...,...,...,...
757,4045,disaster,None,If Locke has to pitch in the playoffs it's not...,0
758,6382,hostages,china,#hot C-130 specially modified to land in a st...,1
759,1001,blazing,None,@dmac1043 Colorado is a Spanish word ([Latin o...,0
760,2332,collapse,Fakefams,Correction: Tent Collapse Story http://t.co/S7...,1


In [6]:
pd.DataFrame(df['test'])

,id,keyword,location,text
0,0,None,None,Just happened a terrible car crash
1,2,None,None,"Heard about #earthquake is different cities, s..."
2,3,None,None,"there is a forest fire at spot pond, geese are..."
3,9,None,None,Apocalypse lighting. #Spokane #wildfires
4,11,None,None,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,None,None,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,None,None,Storm in RI worse than last hurricane. My city...
3260,10868,None,None,Green Line derailment in Chicago http://t.co/U...
3261,10874,None,None,MEG issues Hazardous Weather Outlook (HWO) htt...


Padding adds a special padding token to ensure shorter sequences will have the same length as either the longest sequence in a batch or the maximum length accepted by the model.

Truncation works in the other direction by truncating long sequences.

## DistilBERT

In [7]:
distilBERTtokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Use Predefined Distilbert Tokenizer

def distilBERTtokenize(row):
    return distilBERTtokenizer(row["text"], padding = "max_length", truncation = True) # Padding and Truncation to Max Model Input Length

origin_columns = set(df["train"].features) # Not yet tokenized dataset
encoded_df = df.map(distilBERTtokenize, batched = True) # Open Batch Processing, Default Batch Size is 1000
tokenizer_columns = list(set(encoded_df["train"].features) - origin_columns)
print("Columns added by tokenizer:", tokenizer_columns)

100%|██████████| 1/1 [00:00<00:00,  9.28ba/s]

Columns added by tokenizer: ['attention_mask', 'input_ids']


In [8]:
df

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 6851
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text'],
        num_rows: 3263
    })
    valid: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 762
    })
})

In [9]:
encoded_df

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target', 'input_ids', 'attention_mask'],
        num_rows: 6851
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'input_ids', 'attention_mask'],
        num_rows: 3263
    })
    valid: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target', 'input_ids', 'attention_mask'],
        num_rows: 762
    })
})

In [10]:
data_collator = DataCollatorWithPadding(tokenizer = distilBERTtokenizer, return_tensors = "tf")

train_df = encoded_df['train'].to_tf_dataset(
    columns = tokenizer_columns,
    label_cols = ["target"],
    shuffle = True,
    collate_fn = data_collator,
    batch_size = batch_size,
)

val_df = encoded_df['valid'].to_tf_dataset(
    columns = tokenizer_columns,
    label_cols = ["target"],
    shuffle = False,
    batch_size = batch_size,
    collate_fn = data_collator,
)

test_df = encoded_df['test'].to_tf_dataset(
    columns = tokenizer_columns,
    shuffle = False,
    batch_size = batch_size,
    collate_fn = data_collator,
)

2022-05-31 23:04:14.222887: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-05-31 23:04:14.223307: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-05-31 23:04:14.223674: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-05-31 23:04:14.223709: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2022-05-31 23:04:14.223741: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not lo

<PrefetchDataset element_spec={'input_ids': TensorSpec(shape=(None, None), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, None), dtype=tf.int64, name=None)}>

In [11]:
distilBERT_model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels = 2
)

distilBERT_model.summary()

2022-05-31 23:04:16.228744: W tensorflow/python/util/util.cc:368] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.
Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['vocab_projector', 'vocab_transform', 'vocab_layer_norm', 'activation_13']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint 

Model: "tf_distil_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMai  multiple                 66362880  
 nLayer)                                                         
                                                                 
 pre_classifier (Dense)      multiple                  590592    
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
 dropout_19 (Dropout)        multiple                  0         
                                                                 
Total params: 66,955,010
Trainable params: 66,955,010
Non-trainable params: 0
_________________________________________________________________


In [12]:
num_epochs = 3
batches_per_epoch = len(encoded_df["train"]) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)

optimizer, schedule = create_optimizer(
    init_lr = 2e-5, num_warmup_steps = 0, num_train_steps = total_train_steps
)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True)
distilBERT_model.compile(optimizer = optimizer, loss = loss, metrics = ['accuracy'])

In [13]:
if not os.path.exists(output_dir1): # If the file directory doesn't already exists,
    os.makedirs(output_dir1) # Make it again

checkpoint_callback = ModelCheckpoint(filepath = output_dir1 + '/weights.{epoch:02d}.hdf5', monitor = 'val_loss', save_best_only = True, save_weights_only = True)

In [14]:
distilBERT_model.fit(
    train_df,
    validation_data = val_df,
    epochs = 3,
    callbacks = [checkpoint_callback],
)

Epoch 1/3
428/428 [==============================] - 2746s 6s/step - loss: 0.4288 - accuracy: 0.8153 - val_loss: 0.3878 - val_accuracy: 0.8307
Epoch 2/3
428/428 [==============================] - 2741s 6s/step - loss: 0.3141 - accuracy: 0.8785 - val_loss: 0.4155 - val_accuracy: 0.8202
Epoch 3/3
394/428 [==========================>...] - ETA: 3:32 - loss: 0.2420 - accuracy: 0.9116

### Predict and Output Test Dataset

In [ ]:
test_pred = distilBERT_model.predict(test_df)

In [ ]:
submission = pd.read_csv('dataset/sample_submission.csv')
submission['target'] = np.argmax(test_pred.logits, axis = 1)
submission.to_csv('submission.csv', index = False)

## BERT

In [ ]:
BERTtokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # Use Predefined Distilbert Tokenizer

def BERTtokenize(row):
    return BERTtokenizer(row["text"], padding = "max_length", truncation = True) # Padding and Truncation to Max Model Input Length

origin_columns = set(df["train"].features) # Not yet tokenized dataset
encoded_df = df.map(BERTtokenize, batched = True) # Open Batch Processing, Default Batch Size is 1000
tokenizer_columns = list(set(encoded_df["train"].features) - origin_columns)
print("Columns added by tokenizer:", tokenizer_columns)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer = BERTtokenizer, return_tensors = "tf")

train_df = encoded_df['train'].to_tf_dataset(
    columns = tokenizer_columns,
    label_cols = ["target"],
    shuffle = True,
    collate_fn = data_collator,
    batch_size = batch_size,
)

val_df = encoded_df['valid'].to_tf_dataset(
    columns = tokenizer_columns,
    label_cols = ["target"],
    shuffle = False,
    batch_size = batch_size,
    collate_fn = data_collator,
)

test_df = encoded_df['test'].to_tf_dataset(
    columns = tokenizer_columns,
    shuffle = False,
    batch_size = batch_size,
    collate_fn = data_collator,
)

In [ ]:
BERT_model = TFAutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels = 2
)

BERT_model.summary()

In [ ]:
num_epochs = 3
batches_per_epoch = len(encoded_df["train"]) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)

optimizer, schedule = create_optimizer(
    init_lr = 2e-5, num_warmup_steps = 0, num_train_steps = total_train_steps
)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True)
BERT_model.compile(optimizer = optimizer, loss = loss, metrics = ['accuracy'])

In [ ]:
if not os.path.exists(output_dir2): # If the file directory doesn't already exists,
    os.makedirs(output_dir2) # Make it again

checkpoint_callback = ModelCheckpoint(filepath = output_dir2 + '/weights.{epoch:02d}.hdf5', monitor = 'val_loss', save_best_only = True, save_weights_only = True)

In [ ]:
BERT_model.fit(
    train_df,
    validation_data = val_df,
    epochs = 3,
    callbacks = [checkpoint_callback],
)

In [ ]:
test_pred = BERT_model.predict(test_df)

In [ ]:
submission = pd.read_csv('dataset/sample_submission.csv')
submission['target'] = np.argmax(test_pred.logits, axis = 1)
submission.to_csv('submission2.csv', index = False)